# Read NCIt FLAT — the base graph of the NCI Thesaurus FLAT file (scoped)

`Thesaurus.txt` in the pinned FLAT release zip, as published, for the concepts in scope:

1. pin check of the zip and of `Thesaurus.txt` in it (`Reference_Versions.json`);
2. the file as a text table (tab-separated, no header, no quote handling); the column names from `schema/ncit_flat.yaml`;
3. **the scope** (class annotation `scope`), read from the base graphs of SDTM CT, COSMoS and SDTM Model (`base/sdtm-model/sdtm-model.ttl` is built locally: sign-in content), closed upward over `parents`;
4. the graph of the rows in scope by the CSVW table description: each row node is NCI's concept IRI (aboutUrl template on the code), checked against the published `concept IRI` cell;
5. structural SHACL: must conform;
6. writes `ncit-flat.ttl` (longturtle) and its graph hash;
7. the Excel view `Thesaurus.xlsx`, rebuilt from the graph as written; its text hash must equal that of the source rows in scope;
8. `Provenance.json`, with the scope rule and the graphs it read.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph, URIRef

from cfa import SCHEMA_DIR
from cfa.graph import graph_sha256, write_longturtle
from cfa.pins import checked_source_bytes, table_file
from cfa.provenance import flat, provenance, write_provenance
from cfa.read import read_delimited_text, read_xlsx_table
from cfa.rule import (check_module, columns, csvw_metadata, graph_iri, key_columns, load_module, module_source,
                      pct, property_iri, table_to_graph)
from cfa.view import schema_sheet, text_sha256, view_frame, write_view

MODULE = SCHEMA_DIR / 'ncit_flat.yaml'
TABLE = 'Thesaurus'
sv = load_module(MODULE)
check_module(sv)
SOURCE = module_source(sv)
OUT = REPO_ROOT / 'base' / SOURCE
KEYS = key_columns(sv, TABLE)
SCOPE_RULE = sv.get_class(TABLE).annotations['scope'].value
print('scope:', SCOPE_RULE)

## The generated contract is current

In [ ]:
CSVW_PATH = SCHEMA_DIR / 'csvw' / f'{SOURCE}.csv-metadata.json'
SHACL_PATH = SCHEMA_DIR / 'shacl' / f'{SOURCE}.shacl.ttl'
meta = csvw_metadata(sv, TABLE, f'{pct(TABLE)}.csv')
assert json.loads(CSVW_PATH.read_text(encoding='utf-8')) == meta, 'CSVW not current: run checks/00_Schema'
assert SHACL_PATH.read_text(encoding='utf-8') == ShaclGenerator(str(MODULE)).serialize(), 'SHACL not current: run checks/00_Schema'
print('contract current')

## 1–2. Pin check, the file as a text table

In [ ]:
NAME = table_file(SOURCE, TABLE)
text = checked_source_bytes(SOURCE, NAME).decode('utf-8')
table = read_delimited_text(text, '\t', columns(sv, TABLE), NAME)
assert not table['code'].duplicated().any(), 'a code occurs more than once'
print(NAME, '| rows', len(table), '| columns', len(table.columns))

## 3. The scope, from the SDTM CT, COSMoS and SDTM Model base graphs

In [ ]:
graphs = {s: Graph().parse(REPO_ROOT / 'base' / s / f'{s}.ttl') for s in ['sdtm-ct', 'cosmos', 'sdtm-model']}
values = lambda s, t, c: {str(o) for o in graphs[s].objects(None, URIRef(property_iri(s, t, c)))} - {''}
seeds = {
    'sdtm-ct Code': values('sdtm-ct', 'SDTM Terminology', 'Code'),
    'cosmos ncit_code': values('cosmos', 'cdisc_biomedical_concepts_latest', 'ncit_code'),
    'cosmos ncit_dec_code': values('cosmos', 'cdisc_biomedical_concepts_latest', 'ncit_dec_code'),
    'sdtm-model Variable C-Code': values('sdtm-model', 'SDTM', 'Variable C-Code'),
}
parents_of = {c: [p for p in ps.split('|') if p] for c, ps in zip(table['code'], table['parents'])}
scope = set()
stack = [c for s in seeds.values() for c in s if c in parents_of]
while stack:
    c = stack.pop()
    if c not in scope:
        scope.add(c)
        stack.extend(p for p in parents_of[c] if p not in scope)
unresolved = sorted({p for c in scope for p in parents_of[c] if p not in parents_of})
assert not unresolved, f'parent codes without a row: {unresolved}'
NOT_IN_SOURCE = {name: sorted(c for c in s if c not in parents_of) for name, s in seeds.items()}
scoped = table[table['code'].isin(scope)].reset_index(drop=True)
n_seed = len(set().union(*seeds.values()) & scope)
for name, s in seeds.items():
    print(f'{name}: {len(s)} concepts; without a row: {NOT_IN_SOURCE[name] or "(none)"}')
print(f'scope: {n_seed} concepts and {len(scope) - n_seed} ancestors = {len(scope)} of {len(table)} rows')

## 4–5. The graph (row node = NCI's concept IRI), structural SHACL

In [ ]:
g = table_to_graph(meta, scoped)
p_iri = URIRef(property_iri(SOURCE, TABLE, 'concept IRI'))
mismatch = [(s, o) for s, o in g.subject_objects(p_iri) if str(o) != f'<{s}>']
assert not mismatch, f'row node differs from the published concept IRI: {mismatch[:5]}'
conforms, _, report = pyshacl.validate(g, shacl_graph=Graph().parse(SHACL_PATH))
assert conforms, report
print('triples', len(g), '| every node = its published concept IRI | SHACL conforms')

## 6. Write the graph, its hash

In [ ]:
TTL = OUT / f'{SOURCE}.ttl'
write_longturtle(g, TTL, {'ncit': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#',
                          'flat': f'https://kerfors.github.io/cdisc-for-all/def/{SOURCE}/{pct(TABLE)}#'})
written = Graph().parse(TTL)
GRAPH_HASH = graph_sha256(written)
assert GRAPH_HASH == graph_sha256(g)
print(TTL.name, GRAPH_HASH)

## 7. The Excel view, lossless over the rows in scope

In [ ]:
view = view_frame(written, sv, TABLE)
SOURCE_HASH = text_sha256(scoped, KEYS)
VIEW_HASH = text_sha256(view, KEYS)
assert VIEW_HASH == SOURCE_HASH, 'the view differs from the source rows in scope'
print('text hash, source rows in scope = view:', VIEW_HASH)

## 8. Provenance; write the view and read it back

In [ ]:
prov = provenance(SOURCE, [], [MODULE, CSVW_PATH, SHACL_PATH], graph_iri(SOURCE), len(written),
                  GRAPH_HASH, SOURCE_HASH, VIEW_HASH)
prov['scope'] = {
    'rule': SCOPE_RULE,
    'graphs_read': {s: graph_sha256(gr) for s, gr in graphs.items()},
    'without_row': NOT_IN_SOURCE,
}
write_provenance(OUT / 'Provenance.json', prov)
XLSX = OUT / 'Thesaurus.xlsx'
readme = [
    'NCI Thesaurus, FLAT file (Thesaurus.txt), as published, for the concepts in scope. Derived from the NCI Thesaurus, produced by NCI EVS (CC BY 4.0).',
    f'Scope: {SCOPE_RULE}',
    f'Rebuilt from the graph {TTL.name}: columns in published order (names as in the NCI EVS ReadMe), rows sorted by code.',
    'Every cell is text. The Schema sheet gives each column\'s property IRI; Provenance gives the pin, the scope and the hashes.',
    'cdisc-for-all: https://github.com/kerfors/cdisc-for-all',
]
write_view(XLSX, readme, {TABLE: view}, schema_sheet(sv, [TABLE]), flat(prov))
assert text_sha256(read_xlsx_table(XLSX, TABLE), KEYS) == SOURCE_HASH
print(XLSX.name, 'read back: text hash equal')